# Disaster XAI-Net: six-panel qualitative figure (current pipeline)
For each selected test tile it shows:
(a) the pre-disaster image, (b) the post-disaster image, (c) xBD ground-truth severity,
(d) the **full pipeline**: tuned U-Net footprints coloured by fusion-model severity,
(e) a damaged-building density heatmap drawn **only where buildings exist**, and
(f) rescue-priority zones from the same formula as the dashboard
    (P = w1·D + w2·C + w3·N, 150 m cells, Low/Medium/High/Critical; cells with fewer than 3 buildings are not ranked).

**No models are loaded and nothing is retrained.** The notebook only reads saved outputs. Attach:
1. `lubdham/disaster-separate-data` (test images + labels)
2. the saved output of the **Localization tuning** notebook
   (needs `test_instances_tuned.json` + `end_to_end_predicted_building_severity_tuned.csv`)

CPU is enough (about 1 minute). Download `/kaggle/working/six_panel.zip` from the Output tab when it finishes.

In [ ]:
import os, re, glob, json, math, zipfile
import numpy as np, pandas as pd
from PIL import Image
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPoly, Rectangle, Patch
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter

IN = "/kaggle/input" if os.path.isdir("/kaggle/input") else globals().get("IN_OVERRIDE", ".")
OUT = "/kaggle/working/six_panel" if os.path.isdir("/kaggle/working") else globals().get("OUT_OVERRIDE", "six_panel")
os.makedirs(OUT, exist_ok=True)

TILES = []          # leave empty to auto-pick one representative tile per hazard; or list tile ids, e.g. ["hurricane-harvey_00000123"]
CELL_M = 150.0      # zone size, same default as the dashboard
W = (0.5, 0.35, 0.15); THR = (0.20, 0.35, 0.50); RHO_REF = 400.0; MIN_BLD = 3   # = PriorityConfig defaults
HEAT_SIGMA_M = 25.0  # Gaussian kernel for the heatmap, metres
HEAT_VMAX = 1000.0   # FIXED colour scale for every tile (severity-weighted buildings per km2), so colours are comparable across tiles

def find(pattern):
    h = sorted(glob.glob(os.path.join(IN, "**", pattern), recursive=True)); return h[0] if h else None
INST = find("test_instances_tuned.json"); SEVCSV = find("end_to_end_predicted_building_severity_tuned.csv")
POST_ANY = find("*/test/images/*_post_disaster.png")
assert INST and SEVCSV, "attach the Localization tuning output (test_instances_tuned.json, end_to_end_predicted_building_severity_tuned.csv)"
assert POST_ANY, "attach lubdham/disaster-separate-data"
HAZ = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(POST_ANY))))
print("instances:", INST); print("severity :", SEVCSV); print("images   :", HAZ)

inst = json.load(open(INST))
sev = pd.read_csv(SEVCSV)
SEV_IDX = {"no-damage": 0, "minor-damage": 1, "major-damage": 2, "destroyed": 3}
SEV_COL = {"no-damage": "#1a9850", "minor-damage": "#fee08b", "major-damage": "#f46d43", "destroyed": "#a50026"}
PRI_COL = {"Low": "#4575b4", "Medium": "#fdae61", "High": "#f46d43", "Critical": "#a50026"}

In [ ]:
WKT = re.compile(r"\(\((.*?)\)\)")
def parse(w):
    m = WKT.search(w or "")
    return [tuple(float(v) for v in c.strip().split()[:2]) for c in m.group(1).split(",")] if m else None

def tile_paths(t):
    p = glob.glob(os.path.join(HAZ, "*", "test", "images", f"{t}_post_disaster.png"))
    if not p: return None
    post = p[0]
    return post.replace("_post_disaster.png", "_pre_disaster.png"), post, post.replace("/images/", "/labels/").replace(".png", ".json")

def load_gt(lbl):
    js = json.load(open(lbl)); f = js["features"]
    gt = [{"poly": parse(x["wkt"]), "sev": x["properties"].get("subtype")} for x in f.get("xy", [])]
    gt = [g for g in gt if g["poly"] and g["sev"] in SEV_IDX]
    # metres per pixel from the xy <-> lng_lat correspondence of the same buildings
    ll = {x["properties"].get("uid"): parse(x["wkt"]) for x in f.get("lng_lat", [])}
    src, dst = [], []
    for x in f.get("xy", []):
        p, q = parse(x["wkt"]), ll.get(x["properties"].get("uid"))
        if p and q and len(p) == len(q): src += p; dst += q
    mpp = None
    if len(src) >= 3:
        A = np.c_[np.asarray(src), np.ones(len(src))]
        T = np.linalg.lstsq(A, np.asarray(dst), rcond=None)[0]       # px -> (lon, lat)
        lat0 = float(np.mean(np.asarray(dst)[:, 1]))
        dx = math.hypot(T[0, 0] * 111320 * math.cos(math.radians(lat0)), T[0, 1] * 111320)
        dy = math.hypot(T[1, 0] * 111320 * math.cos(math.radians(lat0)), T[1, 1] * 111320)
        mpp = (dx + dy) / 2
    return gt, mpp

def centroid(poly):
    a = np.asarray(poly); return a[:, 0].mean(), a[:, 1].mean()

def classify(s):
    t1, t2, t3 = THR
    return "Critical" if s >= t3 else "High" if s >= t2 else "Medium" if s >= t1 else "Low"

def zones(blds, size, mpp):
    """blds: list of {cx, cy, sev, p_damaged}. Same arithmetic as backend/prioritization.py (news term N = 0)."""
    # equal cells that tile the image exactly (about CELL_M each), so there are no thin partial cells at the edge
    n = max(1, round(size * mpp / CELL_M)); cpx = size / n; area_km2 = (cpx * mpp / 1000) ** 2
    out = []
    for i in range(n):
        for j in range(n):
            bs = [b for b in blds if j * cpx <= b["cx"] < (j + 1) * cpx and i * cpx <= b["cy"] < (i + 1) * cpx]
            z = {"i": i, "j": j, "x0": j * cpx, "y0": i * cpx, "w": cpx, "n": len(bs)}
            if len(bs) >= MIN_BLD:
                D = np.mean([SEV_IDX[b["sev"]] / 3 for b in bs])
                C = min(1.0, sum(b["p_damaged"] for b in bs) / (area_km2 * RHO_REF))
                P = W[0] * D + W[1] * C
                z.update(D=round(float(D), 3), C=round(float(C), 3), score=round(float(P), 3), cls=classify(P))
            out.append(z)
    return out

In [ ]:
# ---- pick tiles: per hazard, the test tile with many buildings AND a mix of damage classes (ground truth)
def pick_tiles():
    cands = {}
    for lbl in glob.glob(os.path.join(HAZ, "*", "test", "labels", "*_post_disaster.json")):
        t = os.path.basename(lbl).replace("_post_disaster.json", ""); hz = lbl.split(os.sep)[-4]
        if t not in inst: continue
        gt, _ = load_gt(lbl)
        if len(gt) < 20: continue
        c = np.bincount([SEV_IDX[g["sev"]] for g in gt], minlength=4) / len(gt)
        ent = -sum(p * math.log(p) for p in c if p > 0)
        dam = 1 - c[0]
        score = ent * math.log(len(gt)) * (0.3 + dam)
        cands.setdefault(hz, []).append((score, t))
    return [sorted(v)[-1][1] for hz, v in sorted(cands.items())]
tiles = TILES or pick_tiles()
print("tiles:", tiles)

In [ ]:
heat_cmap = LinearSegmentedColormap.from_list("dmg", ["#ffffb2", "#fecc5c", "#fd8d3c", "#f03b20", "#bd0026"])
summary = []
for t in tiles:
    pre_p, post_p, lbl = tile_paths(t)
    gt, mpp = load_gt(lbl)
    post_img = Image.open(post_p).convert("RGB"); pre_img = Image.open(pre_p).convert("RGB")
    size = 1024
    if post_img.size[0] != size:                       # (only for down-sized local copies)
        post_img = post_img.resize((size, size)); pre_img = pre_img.resize((size, size))
    mpp = mpp or 0.5
    pr = sev[sev.tile == t].set_index("pred_building_id")
    preds = []
    for b in inst[t]["buildings"]:
        if b["id"] not in pr.index or not b.get("polygon"): continue
        r = pr.loc[b["id"]]; cx, cy = centroid(b["polygon"])
        preds.append({"poly": b["polygon"], "cx": cx, "cy": cy, "sev": r["severity"], "p_damaged": float(r["p_damaged"])})
    gtb = [{"cx": centroid(g["poly"])[0], "cy": centroid(g["poly"])[1], "sev": g["sev"],
            "p_damaged": 1.0 if SEV_IDX[g["sev"]] > 0 else 0.0} for g in gt]
    Z = zones(preds, size, mpp); Zgt = zones(gtb, size, mpp)

    # ---- heatmap: damaged-building density (per km2), shown only where buildings are
    ds = 4; g = size // ds
    dmg = np.zeros((g, g)); occ = np.zeros((g, g))
    for b in preds:
        x, y = min(g - 1, int(b["cx"] / ds)), min(g - 1, int(b["cy"] / ds))
        dmg[y, x] += SEV_IDX[b["sev"]] / 3; occ[y, x] += 1     # severity weight: no-damage 0, minor 1/3, major 2/3, destroyed 1
    sig = HEAT_SIGMA_M / mpp / ds
    px_km2 = (mpp * ds / 1000) ** 2
    dmg_d = gaussian_filter(dmg, sig) / px_km2; occ_d = gaussian_filter(occ, sig) / px_km2
    alpha = np.clip(occ_d / (0.15 * occ_d.max() + 1e-9), 0, 1) * 0.85 if occ_d.max() > 0 else np.zeros_like(occ_d)
    vmax = HEAT_VMAX
    rgba = heat_cmap(np.clip(dmg_d / vmax, 0, 1)); rgba[..., 3] = alpha

    cnt = lambda L: {k: sum(1 for x in L if x["sev"] == k) for k in SEV_IDX}
    cg, cp = cnt(gt), cnt(preds)
    fmt = lambda c: "  ".join(f"{k.split('-')[0].capitalize()}:{v}" for k, v in c.items())
    fig, ax = plt.subplots(2, 3, figsize=(18, 13.6))
    for a in ax.flat: a.set_xticks([]); a.set_yticks([]); a.set_xlim(0, size); a.set_ylim(size, 0)
    ax[0, 0].imshow(pre_img); ax[0, 0].set_title("(a) Pre-disaster", fontsize=12, weight="bold")
    ax[0, 1].imshow(post_img); ax[0, 1].set_title("(b) Post-disaster", fontsize=12, weight="bold")
    ax[0, 2].imshow(post_img)
    for b in gt: ax[0, 2].add_patch(MplPoly(b["poly"], closed=True, fc=SEV_COL[b["sev"]], ec=SEV_COL[b["sev"]], alpha=0.55, lw=0.8))
    ax[0, 2].set_title(f"(c) Ground truth (xBD), {len(gt)} buildings\n{fmt(cg)}", fontsize=11, weight="bold")
    ax[1, 0].imshow(post_img)
    for b in preds: ax[1, 0].add_patch(MplPoly(b["poly"], closed=True, fc=SEV_COL[b["sev"]], ec=SEV_COL[b["sev"]], alpha=0.55, lw=0.8))
    ax[1, 0].set_title(f"(d) Pipeline: U-Net footprints + fusion severity, {len(preds)} buildings\n{fmt(cp)}", fontsize=11, weight="bold")
    ax[1, 1].imshow(post_img); ax[1, 1].imshow(np.zeros((g, g)), cmap="gray", alpha=0.25, extent=(0, size, size, 0))
    im = ax[1, 1].imshow(rgba, extent=(0, size, size, 0), interpolation="bilinear")
    sm = plt.cm.ScalarMappable(cmap=heat_cmap, norm=plt.Normalize(0, vmax))
    cax = ax[1, 1].inset_axes([1.02, 0.0, 0.035, 1.0]); cb = fig.colorbar(sm, cax=cax); cb.set_label("severity-weighted buildings per km² (fixed scale)", fontsize=9)
    ax[1, 1].set_title(f"(e) Damage heatmap: severity-weighted density (σ = {HEAT_SIGMA_M:.0f} m)\nshown only where buildings were detected", fontsize=11, weight="bold")
    ax[1, 2].imshow(post_img, alpha=0.75)
    for z in Z:
        x0, y0, w = z["x0"], z["y0"], min(z["w"], size - z["x0"])
        h = min(z["w"], size - z["y0"])
        if "cls" in z:
            ax[1, 2].add_patch(Rectangle((x0, y0), w, h, fc=PRI_COL[z["cls"]], ec="white", lw=1, alpha=0.55))
            if w > 60 and h > 40:
                ax[1, 2].text(x0 + w / 2, y0 + h / 2, f"{z['cls']}\n{z['score']:.2f}", ha="center", va="center", fontsize=7.5,
                              color="white", weight="bold")
        else:
            ax[1, 2].add_patch(Rectangle((x0, y0), w, h, fc="none", ec="white", lw=0.6, ls=":", hatch="..", alpha=0.6))
    pc = {c: sum(1 for z in Z if z.get("cls") == c) for c in PRI_COL}
    ax[1, 2].set_title(f"(f) Rescue-priority zones (~{size * mpp / max(1, round(size * mpp / CELL_M)):.0f} m), P = {W[0]}·D + {W[1]}·C\n" +
                       "  ".join(f"{k}:{v}" for k, v in pc.items()) + f"  not ranked:{sum(1 for z in Z if 'cls' not in z)}",
                       fontsize=11, weight="bold")
    handles = [Patch(color=c, label=k.replace("-", " ").capitalize()) for k, c in SEV_COL.items()] + \
              [Patch(color=c, alpha=0.6, label=f"Priority {k}") for k, c in PRI_COL.items()] + \
              [Patch(fc="none", ec="grey", hatch="..", label="Not ranked (< 3 buildings)")]
    fig.legend(handles=handles, loc="lower center", ncol=9, fontsize=9, frameon=False)
    fig.suptitle(f"{t}   (test tile, about {size * mpp:.0f} m across)", fontsize=13)
    fig.subplots_adjust(left=0.01, right=0.97, top=0.92, bottom=0.05, wspace=0.2, hspace=0.2)
    fig.savefig(os.path.join(OUT, f"six_panel_{t}.png"), dpi=150); plt.close(fig)

    # zone agreement with the zones computed from ground truth on the same grid
    both = [(a, b) for a, b in zip(Z, Zgt) if "cls" in a and "cls" in b]
    summary.append({"tile": t, "m_per_px": round(mpp, 3), "gt_buildings": len(gt), "pred_buildings": len(preds),
                    **{f"zones_{k}": v for k, v in pc.items()},
                    "zones_ranked_in_both": len(both),
                    "zone_class_agreement_vs_gt": round(np.mean([a["cls"] == b["cls"] for a, b in both]), 3) if both else None})
    json.dump({"tile": t, "zones_pipeline": Z, "zones_ground_truth": Zgt}, open(os.path.join(OUT, f"zones_{t}.json"), "w"))
    print(summary[-1])

pd.DataFrame(summary).to_csv(os.path.join(OUT, "six_panel_summary.csv"), index=False)
zp = OUT.rstrip("/") + ".zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(OUT): zf.write(os.path.join(OUT, fn), os.path.join("six_panel", fn))
print("wrote", zp)

In [ ]:
from IPython.display import Image as IPImage, display
for t in tiles: display(IPImage(os.path.join(OUT, f"six_panel_{t}.png"), width=1100))